# **Main notebook**

### **I. Engineering financial features for time-series stock data**

- Slice the data into our period of interest (2015-01-01 to 2019-12-31).

Obtain: 
- price
- volume
- SD of Adj Closing prices
- Daily Adj. returns
- Average returns over the previous 3 & 7 days


In [ ]:
import pandas as pd

tickers = ("AAPL", "AMZN", "GOOG", "GOOGL", "MSFT", "TSLA")
df = {}

for ticker in tickers:
    
    # Loading data per ticker
    df_temp = pd.read_csv(f'data/Stock_price/{ticker}.csv')
    df_temp['date'] = pd.to_datetime(df_temp['date'])
    df_temp = df_temp.set_index('date').sort_index()
    
    # Filtering time window: 2015-01-01 to 2019-12-31 (we go back to 2014-12 for sdAdjClose and 7dret)
    df_temp = df_temp.loc['2014-12-01':'2019-12-31']

    df_temp['price'] = df_temp['adj close']
    df_temp['dlyret'] = df_temp['adj close'].pct_change() # Daily adjusted returns
    df_temp['3dret'] = df_temp['dlyret'].rolling(3).mean() # Previous 3 days mean adjusted returns
    df_temp['7dret'] = df_temp['dlyret'].rolling(7).mean() # Previous 7 days mean adjusted returns
    df_temp['sdAdjClose'] = df_temp['adj close'].rolling(20).std() # SD of adjusted closing prices, one trading month

    # We compute metrics only for the period of interest
    df[ticker] = df_temp.loc['2015-01-01':'2019-12-31', ['price', 'volume', 'dlyret', '3dret', '7dret', 'sdAdjClose']]

    # Create individual df: dfAAPL exists. Without this line: df['AAPL'] instead
    globals()[f'df{ticker}'] = df[ticker]

### **II. Reuters News**

In [ ]:
from bs4 import BeautifulSoup
import requests
import time
import pandas as pd

SOURCES = {
    "AAPL":     ("apple-inc",                                    39),
    "AMZN":    ("stream/84cf4073-a674-4a93-aef9-dcc1832a65cb", 33),
    "GOOGL":  ("stream/6656b3e4-1a78-4960-90a0-8d422450f9a6", 11),
    "GOOGL":    ("stream/d6b12f0c-bf3f-4045-a07b-1e4e49103fd6", 32),
    "MSFT": ("stream/4f447b5d-53f5-41bd-ab42-3c0cfc161699", 25),
    "TSLA":     ("stream/35edec46-ef7b-4f9b-b85a-25174e6e07fa", 32),
}

def scrape_ft_topic(slug, start_page, company):
    articles = []
    page = start_page
    while True:
        url = f"https://www.ft.com/{slug}?page={page}"
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")
        items = soup.select(".stream-item[data-id]")

        if not items:
            print(f"\n  [{company}] page {page}: no items, stopping")
            break

        for item in items:
            title_tag = item.select_one(".o-teaser__heading a")
            li = item.find_parent("li")
            time_tag = li.find("time") if li else None
            if not title_tag:
                continue

            date = time_tag["datetime"][:10] if time_tag else ""
            title = title_tag.get_text(strip=True)

            if title.startswith("Lex."):
                title = title[len("Lex."):].strip()

            if date and date[:4] < "2015":
                print(f"\n  [{company}] p{page}: reached {date}, stopping")
                return articles

            if date and date[:4] < "2020":
                articles.append({"company": company, "date": date, "title": title})

        print(f"Scraping {company}: Page {page}, {len(articles)} articles", end="\r")
        page += 1
        time.sleep(1)

    return articles


all_articles = []
for company, (slug, start_page) in SOURCES.items():
    arts = scrape_ft_topic(slug, start_page, company)
    all_articles.extend(arts)

df = pd.DataFrame(all_articles)
df = df.drop_duplicates(subset=["title", "date", "company"])
df = df.sort_values(["company", "date"], ascending=[True, False])

print(f"\nTotal: {len(df)} articles")
print(df.groupby("company").size())
df.to_csv("data/output/HL_FinTimes_raw.csv", index=False)

KeyboardInterrupt: 

We scrape 6387 FT articles headlines: that is less than 1 article per day per firm over the 5-year period we are studying.

### **III. Tweeter sentiment analysis**

- Load tweets
- Define influential tweets
- Compute sentiment: VADER, Bag of Words, TextBlob, roBERTa

We first load tweets: they have already been filtered from spam, and finBERT sentiment has been computed.

In [ ]:
dfTweets = pd.read_csv(f'data/output/FinBERT_sentiment.csv')

#### **III.A. Influential Tweets**

We then define which tweets are influential. A tweet is defined as influential if any of its metrics (comment_num, retweet_num, like_num) exceed the 99th percentile for those metrics. We distinguish influential tweets from non-influential tweets (SHAO et al., 2025).

In [77]:
com99th, rt99th, like99th = dfTweets['comment_num'].quantile(0.99), dfTweets['retweet_num'].quantile(0.99), dfTweets['like_num'].quantile(0.99)

print(f"99th comments: {com99th}\n99th RT: {rt99th}\n99th like: {like99th}")

# Influential: if any of its metric is above the 99th threshold
dfTweets['influential'] = (
    (dfTweets['comment_num'] > com99th) |
    (dfTweets['retweet_num'] > like99th) |
    (dfTweets['like_num'] > like99th)
)

99th comments: 6.0
99th RT: 11.0
99th like: 45.0


#### **III.B. Sentiment analysis**

We then proceed with various types of sentiment analysis: VADER, TextBlob, Bag of Words, roBERTa. 

##### **III.B.1. VADER**

[VADER](https://github.com/cjhutto/vaderSentiment) sentiment analysis. Instead of using the compound score, we substract the negative sentiment scores from positive sentiment scores as our final variable (SHAO et al., 2025).

We compute VADER scores for both tweets and headlines.

In [5]:
# 2min
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

tweets = dfTweets["body"].fillna('').astype(str)
headlines = df["title"].fillna('').astype(str)
analyzer = SentimentIntensityAnalyzer()

In [ ]:
vader_score = []
for tweet in tweets:
    vs = analyzer.polarity_scores(tweet)
    vader_score.append(vs['pos'] - vs['neg'])

dfTweets['VADER'] = vader_score

In [6]:
vader_H_score = []
for headline in headlines:
    vs = analyzer.polarity_scores(headline)
    vader_H_score.append(vs['pos'] - vs['neg'])

df['VADER'] = vader_H_score

##### **III.2.2. Bag of Words (BoW)**

$$\text{BoW score}=
\begin{cases}
\frac{\text{count}_p-\text{count}_n}{\text{count}_p+\text{count}_n}, & \text{if count}_p + \text{count}_n > 0 ;\\
0, & \text{otherwise} ;
\end{cases}
$$

Where $\text{count}_p$ and $\text{count}_n$ denote the count of positive and negative words, respectively.

NLTK initial vocabulary is limited: we add some positive and negative financial jargon to its vocabulary, but the NLKT opinion lexicon still yields discrete sentiment scores (-1, 0, 1) for most tweets due to its limited coverage.

In [ ]:
from nltk.corpus import opinion_lexicon
from nltk.tokenize import word_tokenize
import nltk

# Needed ressources
nltk.download('opinion_lexicon')
nltk.download('punkt')
nltk.download('punkt_tab')

pos_words = set(opinion_lexicon.positive()) # # positive words
neg_words = set(opinion_lexicon.negative())# # negative words

# We add some financial jargon that wasn't in the opinion lexicon
fin_pos = {"surge", "profit", "uptrend", "beat", "buy", "bull", "skyrocket"}
fin_neg = {"plunge", "selloff", "underperform", "downtrend", "sell", "bear", "bankruptcy", "bankrupt"}

pos_words.update(fin_pos)
neg_words.update(fin_neg)

[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     /Users/eyquem/nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to /Users/eyquem/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/eyquem/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# 2min (Tweets)
BoW_scores = []
for tweet in tweets:
    tokens = word_tokenize(tweet.lower())
    count_p = sum(1 for token in tokens if token in pos_words)
    count_n = sum(1 for token in tokens if token in neg_words)

    # BoW score formula
    if count_p + count_n > 0: BoW_score = (count_p - count_n) / (count_p + count_n)
    else: BoW_score = 0.0

    BoW_scores.append(BoW_score)

dfTweets['BoW'] = BoW_scores

In [8]:
BoW_H_scores = []

for headline in headlines:
    Htokens = word_tokenize(headline.lower())
    count_p = sum(1 for Htoken in Htokens if Htoken in pos_words)
    count_n = sum(1 for Htoken in Htokens if Htoken in neg_words)

    # BoW score formula
    if count_p + count_n > 0: BoW_H_score = (count_p - count_n) / (count_p + count_n)
    else: BoW_H_score = 0.0

    BoW_H_scores.append(BoW_H_score)

df['BoW'] = BoW_H_scores

##### **III.B.3. TextBlob (TB)**

We use TextBlob for further sentiment analysis. TB alread provides a sentiment polarity score in the range [-1, 1].

In [9]:
from textblob import TextBlob

In [ ]:
#3m
TB_scores = []

for tweet in tweets:
    TB = TextBlob(tweet)
    TB_scores.append(TB.sentiment.polarity)

dfTweets['TBlob'] = TB_scores
dfTweets.to_csv("data/output/tweets_preRoBERTa.csv")

In [10]:
TB_H_scores = []

for headline in headlines:
    TB_H = TextBlob(headline)
    TB_H_scores.append(TB_H.sentiment.polarity)

df['TBlob'] = TB_H_scores

##### **III.B.4. RoBERTa**

We use a [pre-trained RoBERTa model fine-tuned for sentiment analysis](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest).

Sentiment of the model: 0: = Negative; 1 = Neutral; 2 = Positive

We access the model via HuggingFace.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from scipy.special import softmax
from tqdm import tqdm

# Model config
MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

def preprocess(text):
    if not isinstance(text, str):
        return ""
    new_text = []
    for t in text.split(" "):
        new_text.append(t)
    return " ".join(new_text)

def get_RoBERTa_score(text):
    try:
        text = preprocess(text)
        encoded_input = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        output = model(**encoded_input)
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        
        score = scores[2] - scores[0] # positive - negative
        return score
    except:
        return 0.0

dfTweets = pd.read_csv("data/output/tweets_preRoBERTa.csv")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 33708.72it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
begin, to = 0, 10000
tqdm.pandas(desc="RoBERTa sentiment")
dfTweets['RoBERTa'] = dfTweets['body'].fillna('').astype(str).iloc[begin:to].progress_apply(get_RoBERTa_score)
dfTweets.to_csv(f"data/output/RoBERTa_{begin}_to_{to}.csv", index=True)

RoBERTa sentiment: 100%|██████████| 10000/10000 [04:29<00:00, 37.13it/s]


In [ ]:
tqdm.pandas(desc="RoBERTa sentiment")
df['RoBERTa'] = df['title'].fillna('').astype(str).progress_apply(get_RoBERTa_score)
df.to_csv(f"data/output/HL_sentiment.csv", index=True)

RoBERTa sentiment: 100%|██████████| 6387/6387 [03:33<00:00, 29.98it/s]


#### **III.C. Daily weighted sentiment and influence averages**

$$\text{General Avg}_z=\frac{1}{T}\sum^T_{i=1}S_{i,z}$$
$$\text{Tweet Influential}_z=\frac{\sum^T_{i=1}D_i\times S_{i,z}}{\sum^T_{i=1}D_1}$$
$$\text{Tweet Non-Influential}_z=\frac{\sum^T_{i=1}(1-D_i)\times S_{i,z}}{\sum^T_{i=1}(1-D_i)}$$

Where $S_{i,z}$ denotes the sentiment score of the $i$ th tweet for the $z$ th sentiment type (BoW, Vader, TB, or RoBERTa) and $D_i$ denote the binary flag for tweet influence (either 0 or 1) for the $i$ th tweet.

We cannot do weighted averages since we don't have follower counts to define influential users.

With 3 averages and 3 models, we keep 12 daily time series panels.

#### **REMPLACER DF_SUBSET WHEN WE HAVE PROCESSED EVERYTHING**

In [14]:
df_subset = dfTweets.iloc[:10000].copy()
df_subset['date'] = pd.to_datetime(df_subset['post_date']).dt.date

sentiment_cols = ['VADER', 'BoW', 'TBlob', 'RoBERTa']
all_daily = []

for date, day_df in df_subset.groupby('date'):
    for sentiment in sentiment_cols:
        inf_df = day_df[day_df['influential'] == True]
        non_inf_df = day_df[day_df['influential'] == False]
        
        all_daily.append({
            'Date': date,
            'Model': sentiment,
            'General_Avg': day_df[sentiment].mean(),
            'Influential_Avg': inf_df[sentiment].mean() if len(inf_df) > 0 else None,
            'NonInfluential_Avg': non_inf_df[sentiment].mean() if len(non_inf_df) > 0 else None
        })

daily_df = pd.DataFrame(all_daily).sort_values(['Date', 'Model']).reset_index(drop=True)
print(daily_df.round(4))

         Date    Model  General_Avg  Influential_Avg  NonInfluential_Avg
0  1970-01-01      BoW       0.1295           0.2273              0.1293
1  1970-01-01  RoBERTa       0.1866          -0.1521              0.1874
2  1970-01-01    TBlob       0.0724           0.0340              0.0725
3  1970-01-01    VADER       0.0443           0.0297              0.0444


In [21]:
df['date'] = pd.to_datetime(df['date']).dt.date
daily_df_H = df.groupby('date')[sentiment_cols].mean().reset_index()

print(daily_df_H.round(4))
daily_df_H.to_csv('data/output/HL_sentiment_Dly.csv')

            date   VADER     BoW   TBlob  RoBERTa
0     2015-01-01  0.1480  0.0000  0.5000   0.4863
1     2015-01-04  0.0530  0.5000  0.0219   0.4533
2     2015-01-05  0.0000  0.0000  0.0000  -0.7107
3     2015-01-06  0.0610 -0.1667  0.1500   0.0817
4     2015-01-07  0.0000 -1.0000  0.0000  -0.2160
...          ...     ...     ...     ...      ...
1458  2019-12-19  0.0000  0.0000  0.0000   0.3295
1459  2019-12-23  0.2742  0.8000  0.6400   0.5204
1460  2019-12-27 -0.0390 -0.5000  0.0000   0.0222
1461  2019-12-30  0.0110  0.0000  0.0000   0.0890
1462  2019-12-31 -0.2660  0.0000  0.0000  -0.0626

[1463 rows x 5 columns]


### **III. Methods & models**

#### **III.A. HD-SURDLM (Heterogeneous Dynamic Seemingly Unrelated Regression with Dynamic Linear Models) model**

Pros of this model: 

- takes into account multi-dimensional time-varying features
- cross-sectional stock analysis: correlations and spillover over assets, better than single-stock models

The extended model is articulated as: 

$$\begin{align}
&y_{j,t}=\alpha_J+(\sum^K_{k=1}X_{j,t,k}\beta_{j,t,k}) + Z_{j,t}\gamma_j / u_{j,t} \\
&\beta_{j,t,k}=\beta_{j,t-1,k}+v_{j,t,k}
\end{align}
$$

where $j= 1,... , M$ denotes individual stocks; $t = 1,... , T$ represents discrete time points; and $k= 1,... , K$ indexes over the time-varying features. It comprises stock-specific intercepts $\alpha_j$, time-varying features $X_{j, t, k}$ with coefficients $\beta_{j,t,k}$, and non-time-varying features $Z_{j,t}$ with coefficients $\gamma_j$. The model accounts for observation errors $u_{j,t}\tilde N_M(0,\Omega)$ and system errors $v_{j,t,k}\tilde N(0, \Omega)$, which are independent across time and features, with a covariance structure $\sum = A\Omega A$. The unknown parameters $\Theta = \{\alpha_j,\beta_{j,t,k},\gamma_j,\Omega\}_{j=1,...,M;t=1,...,T}$ collectively capture stock-specific effects, time-varying relationships, and error structures, enhancing the model’s adaptability to complex, dynamic stock return patterns across multiple assets. The model utilizes collected data $D=\{ y_{j,t},X_{j,t,k}, Z_{j,t}\}$, enabling a comprehensive analysis of multi-asset stock returns while accounting for both time-varying and static influences.


 
- $\alpha_j$ serves as the intercept for each stock $j$, epitomizing the intrisinc level of returns in the absence of influence by other variables. 

- The coefficients $\beta_{j,t,k}$ are pivotal, representing the time-varying impact of the $k$ th predictor $X_{j,t,k}$ on the returns of stock $j$ at time $t$ (they may evolve as per $\beta_{j,t,k}=\beta_{j,t-1,k}+v_{j,t,k}$), portraying the temporal variations in the relationships between predictors and returns.

- The model incorporate $\gamma_j$, the set of coefficients corresponding to the predictor vector Z_{j,t} for each stock $j$. Although these coefficient are static across time, their random nature allows fot th emodulation of the the predictors’ impact on returns.

- $\Omega$, encompassed within the unknown parameter vector $\Theta$, is not explicitly defined in the provided context but is potentially indicative of the variance–covariance matrix of the error terms, thereby capturing the variances and correlations in the model residuals.

- The model accommodates error terms representing the observation error and system error, respectively. $u_{j,t}$ accounts for the unexplained variability in stock returns. $v_{j,t,k}$ signifies the fluctuations in the time-varying regression coefficients, thereby contributing to the dynamic nature of the model. 

Collectively, the incorporation of these random parameters enhances the model’s adaptability and proficiency in capturing the multifaceted relationships between various predictors and stock returns across diverse stocks and temporal instances.
